In [1]:
import json
import pandas as pd

with open ("raw_studies.json") as f:
    all_studies = json.load(f)
df_int = pd.read_csv("../data/clean/studies.csv")
int_ids = set(df_int["nctId"])

print(len(all_studies), len(df_int), len(int_ids))

3721 1717 1717


In [2]:
# takinf every study from Romania and its center (site_rows). checking for NaN values among keys

site_rows = []
for s in all_studies:
    ps= s["protocolSection"]
    nct = ps["identificationModule"]["nctId"]
    if nct not in int_ids:
        continue
    for loc in ps.get("contactsLocationsModule", {}).get("locations",{}):
        if loc.get("country") != "Romania":
            continue
        site_rows.append({
            "nctId": nct,
            "facility" : loc.get("facility"),
            "city" : loc.get("city"),
            "state" : loc.get("state"),
            "zip" : loc.get("zip"),
            "lat" : loc.get("geoPoint",{}).get("lat"),
            "lon" : loc.get("geoPoint", {}).get("lon"),
            "status" : loc.get("status")
            })

df_sites = pd.DataFrame(site_rows)

print (df_sites.shape)
print(df_sites["nctId"].nunique())
print(df_sites.isna().sum())

(7524, 8)
1717
nctId          0
facility     179
city           0
state       6078
zip          987
lat          142
lon          142
status      5754
dtype: int64


In [3]:
#how cities are written. nb of distinct city names. firsts and lasts

print(df_sites["city"].nunique())
print(df_sites["city"].str.strip().str.lower().nunique())
print(df_sites["city"].value_counts().head(40))
print(df_sites["city"].value_counts().tail(30))

132
132
city
Bucharest              2383
Cluj-Napoca             977
Timișoara               743
Craiova                 569
Iași                    441
Brasov                  367
Târgu Mureş             303
Oradea                  242
Constanța               174
Baia Mare               124
Sibiu                   124
Ploieşti                 97
Suceava                  83
Galati                   70
Satu Mare                62
Bacau                    62
Floreşti                 58
Piteşti                  54
Ovidiu                   33
Florești                 30
Deva                     28
Buzău                    26
Otopeni                  26
Râmnicu Vâlcea           23
Brăila                   23
Bragadiru                21
Sector 2                 20
Arad                     19
Galați                   17
Targoviste               17
Campulung Muscel         17
Caracal                  16
Baloteşti                13
Alba Iulia               13
Comuna Floresti          12
Mangali

In [4]:
# combining different type of spelling (diacritics, capital letters, blanks)

import unicodedata # mode that knows how diacritics are composed

def city_key(name):
    text = unicodedata.normalize("NFKD", name) #breaks down each base letter
    text = "".join(ch for ch in text if not unicodedata.combining(ch)) #keeps only diacritics, not symbols
    return text.lower().strip()
df_sites["cityKey"] = df_sites["city"].apply(city_key)
print(df_sites["cityKey"].nunique())

summary = df_sites.groupby("cityKey")["city"].agg( rows = "count", spelling = lambda x: " | ".join(sorted(set(x)))) # takes distinct typing, orders and joins them
summary = summary.sort_values("rows",ascending = False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
print (summary)

124
                                rows  \
cityKey                                
bucharest                       2383   
cluj-napoca                      977   
timisoara                        743   
craiova                          569   
iasi                             441   
brasov                           368   
targu mures                      305   
oradea                           242   
constanta                        174   
sibiu                            124   
baia mare                        124   
ploiesti                          97   
floresti                          88   
galati                            87   
suceava                           83   
satu mare                         62   
bacau                             62   
pitesti                           54   
ovidiu                            33   
deva                              28   
otopeni                           26   
buzau                             26   
ramnicu valcea                    23

In [5]:
# defining keys, mapping and cleaning

def city_key(name):
    text = unicodedata.normalize("NFKD", name)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().strip()
    return text.replace("?", "s").replace("\ufffd", "s")
df_sites["cityKey"] = df_sites["city"].apply(city_key)

manual_map = {
    "clujnapoca":                      "Cluj-Napoca",
    "cluj-napoca, cluj":               "Cluj-Napoca",
    "cluj-napoca cluj":                "Cluj-Napoca",
    "clug napoca":                     "Cluj-Napoca",
    "timisoara, timis":                "Timisoara",
    "timisoara, judet timis":          "Timisoara",
    "lasi":                            "Iasi",
    "targu mures, mures":              "Targu Mures",
    "targu-mures, jud. mures":         "Targu Mures",
    "tg mures":                        "Targu Mures",
    "tg. mures":                       "Targu Mures",
    "tg.mures":                        "Targu Mures",
    "tg-mures":                        "Targu Mures",
    "sangeorgiu de mures":             "Sangeorgiu de Mures",
    "sangiorgiu de mures":             "Sangeorgiu de Mures",
    "craiova dolj":                    "Craiova",
    "criaova":                         "Craiova",
    "clarasi":                         "Calarasi",
    "floresti/ cluj":                  "Floresti",
    "comuna floresti":                 "Floresti",
    "comunafloresti":                  "Floresti",
    "balotesti, ilfov":                "Balotesti",
    "baia mare, maramures":            "Baia Mare",
    "targoviste, dambovita":           "Targoviste",
    "oradea, bihor county":            "Oradea",
    "alexandru cel bun":               "Alexandru cel Bun",
    "comuna alexandru cel bun":        "Alexandru cel Bun",
    "sanpetru /brasov":                "Sanpetru",
    "com sanpetru":                    "Sanpetru",
    "resca, com. dobrosloveni":        "Resca",
    "dobrosloveni resca, olt county":  "Resca",
    # can't decide based on name
    "muscel":                          "Campulung Muscel",
    "romania":                         "Unknown city",
    "multiple locations":              "Unknown city",
    "dolj":                            "Unknown city",
    "mures":                           "Unknown city",
    "jud. iasi":                       "Unknown city",
    "jud timis":                       "Unknown city",
    "jud cluj":                        "Unknown city",
    "cluj county":                     "Unknown city",
    "alba":                            "Unknown city",
    "suceagu/ cluj napoca":            "Unknown city",  
    }
def clean_city(key):
    if key.startswith(("buc","sector")):
        return "Bucharest"
    if key in manual_map:
        return manual_map[key]
    return key.title().replace(" De "," de ")

df_sites["cityClean"] = df_sites["cityKey"].apply(clean_city)

In [6]:
# checking city names
print(sorted(df_sites.loc[df_sites["cityClean"] == "Bucharest", "cityKey"].unique()))

review = df_sites.groupby("cityClean").agg(
    rows=("nctId", "count"),
    studies=("nctId", "nunique"),
).sort_values("rows", ascending=False)

print(review.shape[0], review["rows"].sum()) # nb of sites (rows)
print(review.head(15))
print(sorted(review.index))

['bucaresti', 'bucharest', 'bucharesti', 'bucuresti', 'bucureti', 'sector 2', 'sector 3', 'sector 6']
75 7524
             rows  studies
cityClean                 
Bucharest    2421     1201
Cluj-Napoca   990      799
Timisoara     755      607
Craiova       576      487
Iasi          445      388
Brasov        368      311
Targu Mures   319      244
Oradea        243      213
Constanta     174      168
Baia Mare     125      111
Sibiu         124      121
Floresti      104      104
Ploiesti       97       85
Galati         87       78
Suceava        83       83
['Afumati', 'Alba Iulia', 'Alexandru cel Bun', 'Arad', 'Bacau', 'Baia Mare', 'Balotesti', 'Bistrita', 'Botosani', 'Bragadiru', 'Braila', 'Brasov', 'Bucharest', 'Buzau', 'Calarasi', 'Campina', 'Campulung Muscel', 'Caracal', 'Cluj-Napoca', 'Codlea', 'Constanta', 'Cordau', 'Craiova', 'Crisan', 'Deva', 'Dobrosloveni', 'Dorobanti', 'Dumbravita', 'Floresti', 'Focsani', 'Galati', 'Giroc', 'Hunedoara', 'Iasi', 'Leamna de Sus', 'Lugoj',

In [7]:
to_check = df_sites[df_sites["cityClean"] == "TO CHECK"]
print(len(to_check))
print(to_check[["nctId", "facility", "city", "lat", "lon"]].to_string())

0
Empty DataFrame
Columns: [nctId, facility, city, lat, lon]
Index: []


In [8]:
print((df_sites["cityClean"] == "Unknown city").sum()) #cities with no name
print(df_sites["cityClean"].nunique(), len(df_sites)) #clean city names, nb of sites

21
75 7524


In [9]:
# checking if Floresti stands for Floresti, Cluj
# checking faciities names
fl = df_sites[df_sites["cityClean"] == "Floresti"]
print(fl[["lat", "lon"]].describe())

print(df_sites["facility"].nunique())
print(df_sites["facility"].value_counts().head(30))
print(df_sites["facility"].dropna().sample(25, random_state=1).to_string())

             lat        lon
count  61.000000  61.000000
mean   45.384984  25.291120
std     0.723436   0.943331
min    44.513130  23.493750
25%    45.050000  25.696580
50%    45.050000  25.783330
75%    45.050000  25.783330
max    46.745740  25.783330
3571
facility
Research Site                                                                                                                                                                                                                                428
Novartis Investigative Site                                                                                                                                                                                                                  327
GSK Investigational Site                                                                                                                                                                                                                     283
Institutul 

In [10]:
# checking if Floresti stands for Floresti, Cluj, by zip code
print(fl.groupby(["city", "state", "lat", "lon"], dropna=False).size())
print(fl["zip"].str[:2].value_counts(dropna=False))

city             state            lat       lon     
Comuna Floresti  Cluj             NaN       NaN          9
                 NaN              NaN       NaN          3
ComunaFloresti   NaN              NaN       NaN          1
Floresti/ Cluj   NaN              NaN       NaN          3
Floreşti         Cluj             46.74574  23.49375    13
                 NaN              45.05000  25.78333    45
Florești         Cluj             NaN       NaN         27
                 Comuna Floresti  44.51313  25.69658     1
                 NaN              44.51313  25.69658     2
dtype: int64
zip
40      90
None    14
Name: count, dtype: int64


In [11]:
# checking facility names again, since a lot of them have ambiguous names
print(df_sites["facility"].value_counts().head(60).to_string())

fac = df_sites["facility"]
patterns = {
    "research site":          r"^research site",
    "investigat* site":       r"investigat\w* site",
    "clinical trial site":    r"^clinical trial site",
    "local institution":      r"^local institution",
    "site + number":          r"\bsite\s*\d+",
    "additional information": r"^for additional information",
}
for label, pat in patterns.items():
    print(label, fac.str.contains(pat, case=False, regex=True, na=False).sum())

facility
Research Site                                                                                                                                                                                                                                428
Novartis Investigative Site                                                                                                                                                                                                                  327
GSK Investigational Site                                                                                                                                                                                                                     283
Institutul Clinic Fundeni                                                                                                                                                                                                                     60
Clinical Trial Site        

In [12]:
# tagging no name sites
generic_regex = (
    r"research site|investigat\w* site|(?:clinical|study)(?: trial| study)?[\s-]*site"
    r"|local institution|research center\s*\d+$|^site\s*[a-z]*\d+$"
    r"|^for additional information|^several$|^many locations$"
    r"|^multiple locations$|^medical facility$|^[a-z]{1,3}\d+[\d\s]*$"
)

fac = df_sites["facility"]
is_generic = fac.isna() | fac.str.contains(generic_regex, case=False, regex=True, na=False)
print(is_generic.sum(), round(is_generic.mean() * 100, 1))

2841 37.8


In [13]:
# automatic clean facility key 
import re, unicodedata

def facility_key(name):
    if pd.isna(name):
        return None
    s = unicodedata.normalize("NFKD", name) #eliminates diacritics
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    s = s.replace("_", " ")
    s = re.sub(r"\(\s*site[^)]*\)", " ", s)
    s = re.sub(r"\bsite\s*(?:n\.?|nr\.?)?\s*[a-z]*\d+\b", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\b(?:s c|s r l|s a|sc|srl|sa)\b", " ", s)
    s = re.sub(r"\s+\d+$", "", s)
    return re.sub(r"\s+", " ", s).strip()


In [14]:
# most common sites
df_sites["facKey"] = df_sites["facility"].apply(facility_key)
named = df_sites[~is_generic]

top = (
    named.groupby(["cityClean", "facKey"])
    .agg(studies=("nctId", "nunique"), rows=("nctId", "size"), example=("facility", "first"))
    .sort_values("studies", ascending=False)
)
print(len(top))
print(top.head(40).to_string())

1847
                                                                             studies  rows                                                             example
cityClean   facKey                                                                                                                                            
Bucharest   institutul clinic fundeni                                             59    60                                           Institutul Clinic Fundeni
Cluj-Napoca institutul oncologic prof dr ion chiricuta cluj napoca                52    55          Institutul Oncologic "Prof. Dr. Ion Chiricuţă" Cluj Napoca
Bucharest   spitalul clinic colentina                                             52    53                                           Spitalul Clinic Colentina
Craiova     centrul de oncologie sf nectarie                                      52    52             S.C. Centrul de Oncologie Sf. Nectarie SRL ( Site 0394)
Cluj-Napoca medisprof                    

In [15]:
# aliases table
alias = {
    "Fundeni Clinical Institute": ["institutul clinic fundeni", "fundeni clinical institute"],
    "Colentina Clinical Hospital": ["spitalul clinic colentina", "colentina clinical hospital"],
    "Coltea Clinical Hospital": ["spitalul clinic coltea", "coltea clinical hospital"],
    "Oncology Institute Cluj (Chiricuta)": [
        "institutul oncologic prof dr ion chiricuta cluj napoca",
        "institutul oncologic cluj",
        "institutul oncologic prof dr ion chiricuta",
        "prof dr i chiricuta institute of oncology",
    ],
    "Sf. Nectarie Oncology Centre": [
        "centrul de oncologie sf nectarie",
        "centrul de oncologie sfantul nectarie",
        "oncology center sf nectarie",
    ],
    "Neomed Brasov": [
        "centrul medical de diagnostic si tratament ambulator neomed",
        "centrul medical de diagnostic si tratament ambulatoriu neomed",
        "c m d t a neomed",
    ],
    "Regional Institute of Oncology Iasi": [
        "institutul regional de oncologie iasi",
        "institutul regional de oncologie",
    ],
    "Oncomed Timisoara": ["oncomed", "cabinet medical oncomed"],
}

alias_map = {key: name for name, keys in alias.items() for key in keys}

In [16]:
#final column for sites
df_sites["facilityClean"] = df_sites["facKey"].map(alias_map) #checks every key
df_sites["facilityClean"] = df_sites["facilityClean"].fillna(df_sites["facKey"].str.title())
df_sites.loc[is_generic, "facilityClean"] = "Generic site"

In [17]:
named = df_sites[df_sites["facilityClean"] != "Generic site"]
top = (
    named.groupby(["cityClean", "facilityClean"])
    .agg(studies=("nctId", "nunique"))
    .sort_values("studies", ascending=False)
)
print(len(top)) #sites
print(top.head(25).to_string())
print((df_sites["facilityClean"] == "Generic site").sum())

1833
                                                                             studies
cityClean   facilityClean                                                           
Cluj-Napoca Oncology Institute Cluj (Chiricuta)                                  106
Craiova     Sf. Nectarie Oncology Centre                                          98
Bucharest   Fundeni Clinical Institute                                            92
            Colentina Clinical Hospital                                           76
Brasov      Neomed Brasov                                                         60
Iasi        Regional Institute of Oncology Iasi                                   60
Timisoara   Oncomed Timisoara                                                     52
Cluj-Napoca Medisprof                                                             43
Bucharest   Delta Health Care                                                     37
            Spitalul Universitar De Urgenta Bucuresti       

In [18]:
#confidentiality
cand = df_sites[
    (df_sites["facilityClean"] != "Generic site")
    & df_sites["facilityClean"].str.contains(r"\bcmi\b|cabinet medical individual|\bdr\b|\bprof\b", case=False, na=False)
]
print(cand.groupby("facilityClean")["nctId"].nunique().sort_values(ascending=False).head(40).to_string())

facilityClean
Centrul De Gastroenterologie Dr Goldis                                                               17
Spitalul Universitar De Urgenta Militar Central Dr Carol Davila                                      17
Cabinet Medical Dr Geru                                                                              12
Spitalul Clinic De Urgenta Prof Dr Agrippa Ionescu                                                   12
Institutul National De Boli Infectioase Prof Dr Matei Bals                                           11
Cmi Dnbm Dr Pop Lavinia                                                                              11
Institutul Regional De Gastroenterologie Si Hepatologie Prof Dr Octavian Fodor                       11
Prof Dr Alexandru Obregia Psychiatry Hospital                                                        10
Scm Dr Rosu                                                                                          10
Institutul Oncologic Bucuresti Prof Dr Alexandru T

In [19]:
# all CMIs become Private Practice
private_regex = r"\bcmi\b|\bscm\b|private practice|cabinet medical"

private_names = [
    "Centrul De Gastroenterologie Dr Goldis",
    "Dr Goldis Gastroenterology Center",
    "Diabmed Dr Popescu Alexandra",
    "Centrul Medical Endocrinologie Si Diabet Dr Paveliu",
    "Centrul Medical Dr Negrisanu",
]

is_private = (
    df_sites["facilityClean"].str.contains(private_regex, case=False, na=False)
    | df_sites["facilityClean"].isin(private_names)
)
df_sites.loc[is_private, "facilityClean"] = "Private practice"
print(is_private.sum())

160


In [20]:
print(df_sites[["lat", "lon"]].isna().sum())

med = df_sites.groupby("cityClean")[["lat", "lon"]].transform("median") #each city median
df_sites["offDeg"] = ((df_sites["lat"] - med["lat"]) ** 2 + (df_sites["lon"] - med["lon"]) ** 2) ** 0.5 #distance between the point and the median, degrees, not km

known = df_sites["cityClean"] != "Unknown city" #removes unknown cities
far = known & (df_sites["offDeg"] > 0.3)
print(far.sum()) #no coordinates
print(df_sites.loc[far, ["cityClean", "facility", "lat", "lon"]].head(20).to_string()) #coordinates too far away from city 

df_sites["facilityClean"] = df_sites["facilityClean"].str.replace(" De ", " de ", regex=False)

lat    142
lon    142
dtype: int64
34
     cityClean                                          facility       lat       lon
47      Galati                                    Diamed Obesity  45.52281  23.06836
147   Floresti                      Radiotherapy Center Cluj SRL  44.51313  25.69658
173   Floresti  S.C. Radiotherapy Center Cluj S.R.L ( Site 1706)  46.74574  23.49375
190   Floresti      SC Radiotherapy Center Cluj SRL ( Site 2203)  46.74574  23.49375
465   Floresti       Radiotherapy Center Cluj - Medical Oncology  46.74574  23.49375
921     Galati                                    Diamed Obesity  45.52281  23.06836
1232  Floresti                          Radiotherapy Center Cluj  46.74574  23.49375
1269    Galati                                      Milena Sante  45.52281  23.06836
1320  Floresti                                       Andeea Robe  46.74574  23.49375
1600  Floresti                          Local Institution - 0015  44.51313  25.69658
2241  Floresti             

In [21]:
df_sites["coordOk"] = df_sites["lat"].notna() & known & (df_sites["offDeg"] <= 0.3) #has coordinates, city is known, deviation is <0.3

city_coords = (
    df_sites[df_sites["coordOk"]]
    .groupby("cityClean")
    .agg(lat=("lat", "median"), lon=("lon", "median"))
    .reset_index() #sets city as column, not index
)
print(df_sites["coordOk"].sum(), len(city_coords))

print(
    df_sites[df_sites["cityClean"] == "Floresti"]
    .groupby(["lat", "lon"]).size()
    .sort_values(ascending=False).head(5) #how many different coordinates Floresti has. 13 in Cluj, 45 in Prahova, although sites are in Cluj
)

7346 74
lat       lon     
45.05000  25.78333    45
46.74574  23.49375    13
44.51313  25.69658     3
dtype: int64


In [22]:
city_coords.loc[city_coords["cityClean"] == "Floresti", ["lat", "lon"]] = [46.74574, 23.49375] #sets lat and lon for Floresti as in Floresti, Cluj
print(city_coords[city_coords["cityClean"] == "Floresti"])

sites_out = df_sites[["nctId", "cityClean", "facilityClean", "status"]].rename(
    columns={"cityClean": "city", "facilityClean": "facility", "status": "siteStatus"}
)
sites_out.to_csv("../data/clean/sites.csv", index=False)

city_coords = city_coords.rename(columns={"cityClean": "city"})
city_coords.to_csv("../data/clean/city_coords.csv", index=False)

print(sites_out.shape, city_coords.shape)

   cityClean       lat       lon
28  Floresti  46.74574  23.49375
(7524, 4) (74, 3)
